# Customer Retention Intelligence
## 04 — Retention Strategy

### Goal

Convert churn predictions into an actionable customer retention strategy.

### Workflow

1. Load the trained model and selected decision threshold.
2. Score customers by churn probability.
3. Identify high-risk customers.
4. Create risk segments.
5. Analyze the customers targeted for retention.
6. Recommend retention actions based on customer risk profiles.

In [2]:
from pathlib import Path
import joblib
import pandas as pd

MODEL_PATH = Path("../models/churn_gradient_boosting.joblib")
DATA_PATH = Path("../data/processed/telco_churn_clean.csv")

model = joblib.load(MODEL_PATH)
df = pd.read_csv(DATA_PATH)

threshold = 0.35

In [3]:
X = df.drop(columns=["customerID", "Churn"])

df["ChurnProbability"] = model.predict_proba(X)[:, 1]

df["RetentionTarget"] = (
    df["ChurnProbability"] >= threshold
).astype(int)

In [4]:
print("Total customers:", len(df))
print("Customers targeted:", df["RetentionTarget"].sum())
print("Target rate:", f"{df['RetentionTarget'].mean():.1%}")

Total customers: 7043
Customers targeted: 2270
Target rate: 32.2%


In [5]:
df["RiskSegment"] = pd.cut(
    df["ChurnProbability"],
    bins=[0, 0.20, 0.35, 0.60, 1.0],
    labels=["Low", "Moderate", "High", "Very High"],
    include_lowest=True
)

print("=== RISK SEGMENTS ===")
display(
    df["RiskSegment"]
    .value_counts()
    .sort_index()
)

high_risk_customers = (
    df[df["RetentionTarget"] == 1]
    .sort_values("ChurnProbability", ascending=False)
)

print("\n=== TOP RETENTION TARGETS ===")
display(
    high_risk_customers[
        [
            "customerID",
            "ChurnProbability",
            "RiskSegment",
            "tenure",
            "Contract",
            "InternetService",
            "MonthlyCharges",
            "PaymentMethod",
            "TechSupport",
            "OnlineSecurity",
            "Churn"
        ]
    ].head(20)
)

=== RISK SEGMENTS ===


RiskSegment
Low          3753
Moderate     1020
High         1303
Very High     967
Name: count, dtype: int64


=== TOP RETENTION TARGETS ===


,customerID,ChurnProbability,RiskSegment,tenure,Contract,InternetService,MonthlyCharges,PaymentMethod,TechSupport,OnlineSecurity,Churn
2208,7216-EWTRS,0.929220,Very High,1,Month-to-month,Fiber optic,100.80,Electronic check,No,No,Yes
6482,5419-JPRRN,0.917339,Very High,1,Month-to-month,Fiber optic,101.45,Electronic check,No,No,Yes
3380,5178-LMXOP,0.914667,Very High,1,Month-to-month,Fiber optic,95.10,Electronic check,No,No,Yes
4800,9300-AGZNL,0.914667,Very High,1,Month-to-month,Fiber optic,94.00,Electronic check,No,No,Yes
1976,9497-QCMMS,0.914667,Very High,1,Month-to-month,Fiber optic,93.55,Electronic check,No,No,Yes
1704,0107-YHINA,0.911355,Very High,1,Month-to-month,Fiber optic,99.75,Electronic check,No,No,Yes
6866,0295-PPHDO,0.899522,Very High,1,Month-to-month,Fiber optic,95.45,Electronic check,No,No,Yes
2577,4910-GMJOT,0.899522,Very High,1,Month-to-month,Fiber optic,94.60,Electronic check,No,No,Yes
3209,8149-RSOUN,0.899522,Very High,1,Month-to-month,Fiber optic,93.85,Electronic check,No,No,Yes
6240,6521-YYTYI,0.893634,Very High,1,Month-to-month,Fiber optic,93.30,Electronic check,No,No,Yes


In [6]:
def recommend_action(row):
    if row["RiskSegment"] == "Very High":
        if row["Contract"] == "Month-to-month":
            return "Priority outreach + long-term contract incentive"
        return "Priority retention outreach"

    elif row["RiskSegment"] == "High":
        if row["TechSupport"] == "No":
            return "Offer tech support / service assistance"
        return "Targeted retention offer"

    elif row["RiskSegment"] == "Moderate":
        return "Monitor and send engagement offer"

    return "No immediate action"


df["RetentionAction"] = df.apply(recommend_action, axis=1)

In [7]:
df["RetentionAction"].value_counts()

RetentionAction
No immediate action                                 3753
Offer tech support / service assistance             1042
Monitor and send engagement offer                   1020
Priority outreach + long-term contract incentive     967
Targeted retention offer                             261
Name: count, dtype: int64

In [8]:
retention_list = (
    df[df["RetentionTarget"] == 1]
    .sort_values("ChurnProbability", ascending=False)
    [
        [
            "customerID",
            "ChurnProbability",
            "RiskSegment",
            "tenure",
            "Contract",
            "MonthlyCharges",
            "TechSupport",
            "OnlineSecurity",
            "PaymentMethod",
            "RetentionAction"
        ]
    ]
)

retention_list.head(20)

,customerID,ChurnProbability,RiskSegment,tenure,Contract,MonthlyCharges,TechSupport,OnlineSecurity,PaymentMethod,RetentionAction
2208,7216-EWTRS,0.929220,Very High,1,Month-to-month,100.80,No,No,Electronic check,Priority outreach + long-term contract incentive
6482,5419-JPRRN,0.917339,Very High,1,Month-to-month,101.45,No,No,Electronic check,Priority outreach + long-term contract incentive
3380,5178-LMXOP,0.914667,Very High,1,Month-to-month,95.10,No,No,Electronic check,Priority outreach + long-term contract incentive
4800,9300-AGZNL,0.914667,Very High,1,Month-to-month,94.00,No,No,Electronic check,Priority outreach + long-term contract incentive
1976,9497-QCMMS,0.914667,Very High,1,Month-to-month,93.55,No,No,Electronic check,Priority outreach + long-term contract incentive
1704,0107-YHINA,0.911355,Very High,1,Month-to-month,99.75,No,No,Electronic check,Priority outreach + long-term contract incentive
6866,0295-PPHDO,0.899522,Very High,1,Month-to-month,95.45,No,No,Electronic check,Priority outreach + long-term contract incentive
2577,4910-GMJOT,0.899522,Very High,1,Month-to-month,94.60,No,No,Electronic check,Priority outreach + long-term contract incentive
3209,8149-RSOUN,0.899522,Very High,1,Month-to-month,93.85,No,No,Electronic check,Priority outreach + long-term contract incentive
6240,6521-YYTYI,0.893634,Very High,1,Month-to-month,93.30,No,No,Electronic check,Priority outreach + long-term contract incentive


## Retention Strategy Summary

Using a decision threshold of **0.35**, the model identifies **2,270 customers (32.2%)** as potential retention targets.

The highest-risk customers are strongly characterized by short tenure, month-to-month contracts, fiber optic service, relatively high monthly charges, electronic check payments, and lack of technical support or online security.

Rather than applying the same retention strategy to every customer, predictions can be translated into targeted actions:

- **Very High Risk:** prioritize immediate outreach and offer incentives for longer-term contracts.
- **High Risk:** provide targeted retention offers and address missing support services.
- **Moderate Risk:** monitor customer behavior and use lower-cost engagement campaigns.
- **Low Risk:** no immediate retention intervention is required.

This analysis demonstrates how churn probabilities can be converted into a prioritized customer retention workflow rather than being used only as classification outputs.

> **Note:** This notebook scores the historical dataset as a demonstration of the retention workflow. In a production setting, the trained pipeline would score current customers whose future churn outcome is not yet known.

In [9]:
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

retention_list.to_csv(
    OUTPUT_DIR / "retention_target_list.csv",
    index=False
)